# Floatmatcher quickstart

`set_points` → `set_product` → `set_method` → `match` → `to_xarray`/`to_dataframe`

In [ ]:
import numpy as np
import pandas as pd
import cdsapi
import matplotlib.pyplot as plt

from floatmatcher.floatmatcher import FloatMatcher

### Points

In [ ]:
# a few positions in the North Atlantic, early January 2018
df = pd.DataFrame({
    "longitude": [-38.2, -35.6, -31.9, -27.4, -23.1],
    "latitude":  [31.5, 34.2, 37.8, 40.1, 43.6],
    "date": pd.to_datetime(["2018-01-02 03:10", "2018-01-03 14:45", "2018-01-05 08:20",
                            "2018-01-06 21:05", "2018-01-08 11:30"]),
})

my_fm = FloatMatcher()
my_fm.set_points(df)  # columns longitude, latitude, date by default
df

### Product

In [ ]:
my_fm.set_product(type="era5-hourly",
                  source="remote-cds",                # downloaded from the Copernicus Climate Data Store
                  path="./cache",                     # cache directory, reused between sessions
                  selected_variables=["sst"],
                  client=cdsapi.Client(),             # key in ~/.cdsapirc, see https://cds.climate.copernicus.eu/how-to-api
                  time=["00:00", "06:00", "12:00", "18:00"],  # hours to download, all of them by default
                  )

# local files instead:
# my_fm.set_product(type="era5-hourly", source="local", path="/<era5_folder>", selected_variables=["sst"])

### Method

In [ ]:
my_fm.set_method(type="nearest",
                 radius=25,                         # km
                 time_gap=np.timedelta64(6, "h"),
                 )

### Matchup

In [ ]:
my_fm.match()  # downloads what is missing in the cache, then matches
my_fm.result

### Results

In [ ]:
matched = my_fm.to_dataframe()  # the input DataFrame, with sst_coloc added
matched

In [ ]:
ax = matched.plot.scatter(x="longitude", y="latitude", c="sst_coloc", cmap="viridis")
ax.set_title("sst at the points (K)")
plt.show()

### Directly from argopy

An argopy dataset is matched in one call, with the `floatmatcher` extension of its `.argo` accessor: the dataset comes back with the `_coloc` variables, in its layout (points or profiles).

In [ ]:
from argopy import DataFetcher

ds = DataFetcher().region([-28, -25, 48, 50, 0, 50, "2018-01", "2018-02"]).data

ds_matched = ds.argo.floatmatcher.match(
                product=dict(type="era5-hourly", source="remote-cds", path="./cache",
                            selected_variables=["sst"], client=cdsapi.Client(),
                            time=["00:00", "06:00", "12:00", "18:00"]),
                method=dict(type="nearest", radius=25, time_gap=np.timedelta64(6, "h")),
            )